Cell 1 - Load the Brain Surface
The brain surface (cortex) is a thin, crumpled sheet - like a piece of paper scrunched. It has two sides:
Outer surface = pial surface (where the brain meets fluid)
Inner surface = white matter surface (where gray matter meets white matter)

The best representation is the middle of these two — called the midthickness surface. This is what Ribeiro et al. use.
fsaverage used - the "average brain" built from hundreds of real people's MRI scans. Standard reference brain everyone uses in neuroscience (FreeSurfer, Fischl 1999).

In [1]:
from nilearn import datasets, surface
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path, connected_components
import torch
from torch_geometric.data import Data
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# fsaverage = standard average brain atlas (FreeSurfer, Fischl et al. 1999)
# midthickness = midpoint between white matter and pial surface
# → the surface Ribeiro et al. 2021 use for deepRetinotopy
fsaverage = datasets.fetch_surf_fsaverage('fsaverage')

coords_pial,  faces = surface.load_surf_mesh(fsaverage.pial_left) # outer surface
coords_white, _     = surface.load_surf_mesh(fsaverage.white_left) # inner surface

# Compute midthickness as the coordinate average (Ribero uses midthickness)
coords = (np.array(coords_pial) + np.array(coords_white)) / 2.0   # middle = midthikness -> 163,000 points in 3D space. Each point = spot on the brain surface
faces  = np.array(faces)                                            # triangles connecting those points. surface is made of ~326,000 tiny triangles -> like a 3D mesh

print(f"  Vertices : {coords.shape[0]:,}")
print(f"  Faces    : {faces.shape[0]:,}")
# MNI coordinates — mm from the center (0, 0, 0)
# x: Left <-> Right - min negative value cause left side of the brain, max value (a few points on the inner wall that touch the midline -> where left and right hemispheres meet) -> absolute value is total width
print(f"  X range  : {coords[:,0].min():.1f} → {coords[:,0].max():.1f} mm")
# y: Back <-> Front - min negative value back of the brain (anchor the fovea) and max positive value front of the brain -> absolute value is total length
print(f"  Y range  : {coords[:,1].min():.1f} → {coords[:,1].max():.1f} mm")
# z: Bottom <-> Top - min negative value below center (base of brain) and max positive value top of the brain (crown) -> absolute value is total height
print(f"  Z range  : {coords[:,2].min():.1f} → {coords[:,2].max():.1f} mm")

c:\Users\laraj\anaconda3\envs\neuroai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[fetch_surf_fsaverage] Dataset found in C:\Users\laraj\nilearn_data\fsaverage
  Vertices : 163,842
  Faces    : 327,680
  X range  : -67.2 → 1.8 mm
  Y range  : -103.7 → 66.9 mm
  Z range  : -44.8 → 76.6 mm


Cell 2 - Isolate the Visual Cortex
The visual cortex sits at the very back of the head - occipital lobe. In MRI coordinates, "back" means negative y-axis. So everything with y < -70 mm is roughly the visual part of the brain.
Don't use the whole brain to analyze - just the part that processes vision. This is the mask.

In [2]:
visual_mask = coords[:, 1] < -70       # posterior occipital lobe in MNI y-axis
visual_idx  = np.where(visual_mask)[0] # ~17,000 vertices that belong to visual cortex (out of 163,000 total)
coords_vis  = coords[visual_idx]       # 3D coordinates of just those visual_idx - ~10.5% of the brain surface (realistic size for visual cortex)

print("Visual-cortex vertices:", coords_vis.shape)   # expect ~ (17263, 3)
print("Fraction of brain kept :", round(100 * len(visual_idx) / len(coords), 1), "%")

Visual-cortex vertices: (17263, 3)
Fraction of brain kept : 10.5 %


Cell 3 - Find the Fovea + Polar Angle
Vision has a center point - main field of sight. That's the fovea in the eye. On the brain, the fovea is represented at the occipital pole - the very tip at the back of the head (most posterior point, lowest y value (negative)).
Polar angle is like a compass direction in the visual field. Two metrics to find stimulus in visual field: eccentricity (how far from center), polar angle (is the stimulus above, below, left, or right of where you're looking).

In [3]:
occipital_pole_idx = int(np.argmin(coords_vis[:, 1])) # most posterior = fovea
occipital_pole     = coords_vis[occipital_pole_idx] # its 3D coordinates

# Compute polar angle around the pole 
# Polar angle  = which DIRECTION  (using x and z coordinates)
# Eccentricity = how FAR from center (calculated later with different distance calculations) 
#     -> can't just use straight-line distance in x/z cause distance measured along the brain surface (geodesic)

dx          = coords_vis[:, 0] - occipital_pole[0]   # left-right offset from pole
dz          = coords_vis[:, 2] - occipital_pole[2]   # up-down offset from pole
polar_angle = np.arctan2(dz, dx)                     # arctan2 turns (dx, dz) into a compass direction (the polar angle), range [−π, π]

print("Occipital pole index :", occipital_pole_idx)
print("Occipital pole (mm)  :", np.round(occipital_pole, 1))
print("Polar angle range    :", round(polar_angle.min(), 2), "to", round(polar_angle.max(), 2), "rad")

Occipital pole index : 1359
Occipital pole (mm)  : [ -12.2 -103.7    1.6]
Polar angle range    : -3.14 to 3.14 rad


Cell 4 - Build the Graph + Euclidean Edges
Convert it into a graph - a network of nodes connected by edges. Each vertex is a node. Each shared edge between two triangles becomes a graph edge. The edge weight = the physical straight-line distance between two neighboring points on the brain surface (in mm).
This is the Euclidean (flat) distance - the baseline before we applying any curvature

In [4]:
# Map original brain vertex IDs -> new compact 0..N-1 IDs
old_to_new = {old: new for new, old in enumerate(visual_idx)}

# Collect every triangle edge whose BOTH endpoints are in visual cortex
edge_set = set()
for tri in faces:       
    if tri[0] in old_to_new and tri[1] in old_to_new and tri[2] in old_to_new:
        a_l = old_to_new[tri[0]]
        b_l = old_to_new[tri[1]]
        c_l = old_to_new[tri[2]]
        for u, v in [(a_l, b_l), (b_l, c_l), (a_l, c_l)]:
            edge_set.add((min(u, v), max(u, v))) # store each edge once

# lists which vertices are connected
edges    = np.array(list(edge_set)).T   # shape (2, n_unique_edges) - source and target indices
u_idx    = edges[0]
v_idx    = edges[1]

# Straight-line (Euclidean) length of each edge, in mm
diff              = coords_vis[u_idx] - coords_vis[v_idx]
euclidean_weights = np.linalg.norm(diff, axis=1)   # distance between two neighboring vertices

print("Unique edges        :", edges.shape[1])                       # expect ~51,216
print("Edge length min/mean/max (mm):",
      round(euclidean_weights.min(), 2),
      round(euclidean_weights.mean(), 2),
      round(euclidean_weights.max(), 2))

Unique edges        : 51216
Edge length min/mean/max (mm): 0.24 0.79 1.64


Cell 5 - Geodesic Distance from the Fovea
To model retinotopy, one need to know how far each point on the visual cortex is from the foveal representation - but measured along the brain surface, not through the brain tissue.
This is much more meaningful than straight-line (Euclidean) distance because signals travel along the cortical sheet, not through it.

In [6]:
from scipy.sparse.csgraph import dijkstra

# Build a sparse graph (only neighboring connections = edge length)
N   = len(visual_idx)
row = np.concatenate([u_idx, v_idx])
col = np.concatenate([v_idx, u_idx])
dat = np.concatenate([euclidean_weights, euclidean_weights])
adj = sp.csr_matrix((dat, (row, col)), shape=(N, N))

# Shortest path along the surface, starting ONLY from the pole
geo_from_pole = dijkstra(adj, directed=False, indices=occipital_pole_idx)

print("geo_from_pole shape :", geo_from_pole.shape)                  # expect (17263,)
print("distance min/max(mm):", round(geo_from_pole.min(), 1), "to", round(geo_from_pole.max(), 1))

geo_from_pole shape : (17263,)
distance min/max(mm): 0.0 to 77.7


Cell 6 - Retinotopy: Eccentricity + Magnification
Two concepts:
Eccentricity = how far from the center of vision, measured in degrees. Looking straight ahead = 0°. Glancing to the side = 10°, 20°, etc. The fovea handles 0–5°, periphery handles up to ~90°.

Cortical magnification = the brain devotes MORE surface area to the center of your vision than the periphery. If you look at the brain with a ruler, 1 mm of cortex near the fovea represents just 0.1° of visual angle, but 1 mm in the periphery represents 5°+ of visual angle. This is why your central vision is so sharp — more brain tissue per degree.
The relationship follows a log-polar model (Schwartz 1980): eccentricity grows exponentially with distance from the occipital pole.

In [7]:
# Parameters - all fixed from literature, do not change 
a_param = 0.5    # deg   - small offset so the fovea starts near 0 deg
                 #         Schwartz 1977 (Biol Cybern); Benson et al. 2018 (J Neurosci)
scale   = 15.0   # mm    - controls how fast eccentricity grows with distance
                 #         human V1 range is 10–17 mm; 15 is the central estimate
                 #         Horton & Hoyt 1991 (Brain); Sereno et al. 1995 (Science)
k_param = 0.065  # deg⁻¹ - controls the magnification fall-off M = 1/(k·ecc + a)
                 #         Horton & Hoyt 1991 (Brain)

cortical_r   = geo_from_pole                       # surface distance from pole (mm)
eccentricity = np.exp(cortical_r / scale) - a_param  # log-polar model
eccentricity = np.clip(eccentricity, 0.1, 12.0)      # keep in a realistic range

# Cortical magnification factor (Horton & Hoyt 1991 style):
#   high near the fovea, low in the periphery
# Source: Horton & Hoyt 1991 (Brain)
magnification = 1.0 / (k_param * eccentricity + a_param)  # (N,) mm²/deg²

# Visual-field position (in degrees) that each vertex represents:
#   polar coordinates (eccentricity, polar_angle) -> Cartesian (x, y)
node_x = eccentricity * np.cos(polar_angle)
node_y = eccentricity * np.sin(polar_angle)

print("eccentricity  min/max:", round(eccentricity.min(), 2), "to", round(eccentricity.max(), 2), "deg")
print("magnification min/max:", round(magnification.min(), 3), "to", round(magnification.max(), 3))

eccentricity  min/max: 0.5 to 12.0 deg
magnification min/max: 0.781 to 1.878


Cell 7 - pRF Responses (Simulated fMRI Signal)
pRF = population Receptive Field. Each small patch of visual cortex responds to a specific region of the visual field. When a stimulus appears there, that patch activates. When the stimulus is elsewhere, it stays quiet.
The response isn't all-or-nothing - it follows Gaussian (bell curve): strongest response at the preferred location, weaker for nearby stimuli, zero for far-away ones. The width of this bell curve (σ) is larger in the periphery -> peripheral cortex is less precise about exactly where the stimulus is.
Simulate 100 different stimuli placed at different visual field positions and record how each of the 17,263 cortical vertices would respond -> like real fMRI experiment.

In [8]:
np.random.seed(42)   # reproducible stimuli + noise

S = 100              # number of stimuli
# Fixed parameters — from literature 
# Receptive field size: σ = 0.2 + 0.4 × eccentricity
# Linear scaling with eccentricity from Harvey & Dumoulin 2011 (J Neurosci):
#   σ ≈ 0.2° at the fovea, growing to ~4° at 10° eccentricity
# This encodes cortical magnification in the response model:
#   tight responses near fovea = high precision representation
#   broad responses in periphery = coarse representation
sigma = 0.2 + 0.4 * eccentricity    # (N,) degrees — one σ per node (pRF size grows with eccentricity)

np.random.seed(42)    # fixed seed for reproducibility
S = 100               # number of stimuli — enough for proof-of-concept

# Scatter 100 stimuli across the visual field (eccentricity 0-10 deg, any angle)
stim_ecc = np.random.uniform(0.0, 10.0, S)
stim_ang = np.random.uniform(-np.pi, np.pi, S)
stim_x   = stim_ecc * np.cos(stim_ang)
stim_y   = stim_ecc * np.sin(stim_ang)

# pRF size grows with eccentricity
sigma = 0.2 + 0.4 * eccentricity

# For each stimulus, compute every vertex's Gaussian response
responses = np.zeros((len(visual_idx), S))
for s in range(S):
    dist_vf = np.sqrt((node_x - stim_x[s])**2 + (node_y - stim_y[s])**2)
    responses[:, s] = np.exp(-0.5 * (dist_vf / sigma)**2)

# Add small measurement noise to mimic real fMRI
responses += 0.01 * np.random.randn(*responses.shape)

print("responses shape     :", responses.shape)            # expect (17263, 100)
print("response value range:", round(responses.min(), 2), "to", round(responses.max(), 2))

responses shape     : (17263, 100)
response value range: -0.04 to 1.02


In [9]:
# SAVING
import os
import torch
from torch_geometric.data import Data
from pathlib import Path

save_dir = Path.cwd()
os.makedirs(save_dir, exist_ok=True)
retino_path = os.path.join(save_dir, "retinotopy_groundtruth.pt")
curved_path = os.path.join(save_dir, "curved_metrics_kappa.pt")   # used in Cell 9

# columns: [x, y, z, eccentricity, polar_angle, magnification]
node_features = np.column_stack([coords_vis, eccentricity, polar_angle, magnification])

# Edge index: make the graph BIDIRECTIONAL (u->v and v->u)
ei = np.vstack([np.concatenate([u_idx, v_idx]),
                np.concatenate([v_idx, u_idx])])   # shape (2, 102432)

# Build the Data object (the map + substrate + signal)
data_gt = Data(
    x               = torch.tensor(node_features, dtype=torch.float32),
    edge_index      = torch.tensor(ei, dtype=torch.long),
    pos             = torch.tensor(coords_vis, dtype=torch.float32),
    y               = torch.tensor(responses, dtype=torch.float32),
    eccentricity_gt = torch.tensor(eccentricity.astype(np.float32)),
    polar_angle_gt  = torch.tensor(polar_angle.astype(np.float32)),
)

torch.save({
    'data': data_gt,
    'meta': {
        'layer': 1,
        'content': 'ground-truth retinotopic map + graph + pRF training signal',
        'n_nodes': int(N),
        'n_directed_edges': int(ei.shape[1]),
        'occipital_pole_idx': occipital_pole_idx,
        'note': 'Same answer key for every kappa. Pair with curved_metrics_kappa.pt.',
    }
}, retino_path)

print(" data saved ")
print("  x        :", tuple(data_gt.x.shape))
print("  edge_index:", tuple(data_gt.edge_index.shape))
print("  y        :", tuple(data_gt.y.shape))
print("  ecc_gt   :", tuple(data_gt.eccentricity_gt.shape))


 data saved 
  x        : (17263, 6)
  edge_index: (2, 102432)
  y        : (17263, 100)
  ecc_gt   : (17263,)
